The goal of this notebook is to precompute ST cell counts and estimated singlecell cell counts per voxel, and save the output as coords_ext.csv

In [101]:
import warnings
import os 
from pathlib import Path
import zarr
import json
from tqdm import tqdm

import pandas as pd
import scanpy as sc
import numpy as np
from dotenv import dotenv_values
from matplotlib import pyplot as plt

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [ ]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "11_brainrender"
os.makedirs(OUTDIR, exist_ok=True)

OBJECTS_DIR = DATADIR / "9_objects" 
HM_VOXEL_FP = OBJECTS_DIR / "spatial" / 'hypomap3d_spatial_voxel_level.zarr'
HM_ST_SC_FP = OBJECTS_DIR / "spatial" / 'hypomap3d_spatial_cell_level.h5ad'
HM_SC_FP = OBJECTS_DIR / "singlecell" / 'hypomap3d_singlecell_all.h5ad'
STRUCTURE_TREE_FP = OBJECTS_DIR / "spatial" / "ABC_structure_tree_processed.json"

COORD_FP = OUTDIR / "coord_space.csv"
COORD_FP_OUT = OUTDIR / "coord_space_ext.csv"


In [14]:
# Notebook constants
ROOT_REGION_ID = 997
LEVEL_OF_ANALYSIS = "L4" 

In [ ]:
voxel_store = zarr.open(HM_VOXEL_FP)
adata_st = sc.read_h5ad(HM_ST_SC_FP, backed="r")
adata_sc = sc.read_h5ad(HM_SC_FP, backed="r") 

coords = pd.read_csv(COORD_FP)

In [49]:
with open(STRUCTURE_TREE_FP, 'r') as fp:
    structure_tree = json.load(fp)

subregions = structure_tree['1097']['subregions'] #HY subregions

### Let's precompute ST cell numbers per voxel 

In [17]:
# Extract root mask and derive voxel shape from it
idx = list(voxel_store['region']['ids']).index(ROOT_REGION_ID) # root region (all region encompassing)
root_mask = voxel_store['region']['region_masks'][idx, ...]

nx, ny, nz = root_mask.shape
n_voxels = nx * ny * nz

print((nx, ny, nz), n_voxels)

(168, 144, 107) 2588544


In [85]:
cell_coords = adata_st.obsm['coords_voxels_25um_bounded'].values
ravel_coords = np.ravel_multi_index(cell_coords.T, dims=(nx, ny, nz))

# Construct counts grid
total_counts = np.bincount(
    ravel_coords,
    minlength=n_voxels
).astype(np.int32)

coords["ST_cell_counts"] = total_counts
coords["is_HY_structure"] = coords.allen_id.isin(subregions)

### Let's estimate singlecell cell counts per voxel 

In [ ]:
ids = [x.item() for x in voxel_store[LEVEL_OF_ANALYSIS]["ids"]]
id_to_idx = {id_: i for i, id_ in enumerate(ids)}

ct_counts = adata_sc.obs[f"{LEVEL_OF_ANALYSIS}_ID"].value_counts()
indices = np.array([id_to_idx[ct] for ct in ct_counts.index])
ct_counts = ct_counts.to_numpy()

rel_mass = voxel_store[LEVEL_OF_ANALYSIS]["rel_mass"][indices].reshape(len(indices), -1)
rel_mass = np.nan_to_num(rel_mass, nan=0.0)

total_counts_sc = (rel_mass * ct_counts[:, None]).sum(axis=0)#.astype(np.int32)

coords["estimated_sc_cell_counts"] = total_counts_sc

In [118]:
coords

,x_ccf,y_ccf,z_ccf,allen_id,allen_acronym,is_inside_HY,ST_cell_counts,is_HY_structure,estimated_sc_cell_counts
0,4375.000000,3950.000048,3049.999952,672,CP,False,0,False,0.113910
1,4375.000000,3950.000048,3075.000048,672,CP,False,0,False,0.120328
2,4375.000000,3950.000048,3100.000143,672,CP,False,1,False,0.126162
3,4375.000000,3950.000048,3125.000238,672,CP,False,0,False,0.131192
4,4375.000000,3950.000048,3150.000334,672,CP,False,0,False,0.135178
...,...,...,...,...,...,...,...,...,...
2588539,8550.015926,7525.013685,5600.009680,0,NaN,False,0,False,0.000000
2588540,8550.015926,7525.013685,5625.009775,0,NaN,False,0,False,0.000000
2588541,8550.015926,7525.013685,5650.009871,0,NaN,False,0,False,0.000000
2588542,8550.015926,7525.013685,5675.009966,0,NaN,False,0,False,0.000000


In [117]:
coords.to_csv(COORD_FP_OUT, index=False)